# 2. Microsoft Defender for Cloud — Implementation

Defender for Cloud is Azure's built-in **CSPM + CWPP** platform:

* **CSPM** — Cloud Security Posture Management: finds misconfigurations and compliance gaps across your estate (including AWS and GCP).
* **CWPP** — Cloud Workload Protection Platform: runtime protection for VMs, SQL, storage, containers, Key Vault, App Service, etc.

### What you'll build in this notebook

1. A **Secure Score calculator** that mirrors how Microsoft weights findings.
2. A **Defender plan advisor** — given a workload, which Defender plan you should enable and why.
3. A **monthly cost estimator** for Defender plans.
4. A **regulatory compliance report** over a simulated subscription.
5. A **bad → best** rollout of Defender for Cloud across a subscription.

No Azure subscription required — everything is Python.

## Before you run this notebook

1. From the lab folder run `uv sync`.
2. In VS Code, pick the `.venv` kernel (top-right kernel picker).
3. If the kernel doesn't appear, reload the window (`Cmd+Shift+P` → *Developer: Reload Window*).

## 1. CSPM vs CWPP — what's actually different?

| Question                                 | CSPM (posture)                 | CWPP (workload)                 |
|------------------------------------------|--------------------------------|---------------------------------|
| When does it run?                        | Continuously, config-time      | Runtime, at the time of the event |
| What does it produce?                    | Recommendations & Secure Score | Security alerts                 |
| What does it need to read?               | ARM, Policy, resource config   | Logs, file events, process events |
| Example                                  | "Storage firewall is Allow All" | "Crypto-mining malware on vm-01" |

Both ship in Defender for Cloud. The **free** tier gives you the foundational CSPM. Paid **Defender plans** add CWPP per workload type *and* premium CSPM features (attack path analysis, agentless scanning).

In [ ]:
# A tiny classifier: is a given finding CSPM (posture) or CWPP (runtime)?
FINDINGS = [
    ("Storage account allows public blob access",                    "CSPM"),
    ("Suspicious PowerShell downloaded encoded payload on vm-web-1", "CWPP"),
    ("Subnet 'snet-db' has no NSG attached",                         "CSPM"),
    ("SQL database 'db-finance' hit by SQL injection pattern",       "CWPP"),
    ("Key Vault 'kv-prod' has no purge protection",                  "CSPM"),
    ("Container image runs as root and shells out to /bin/sh",       "CWPP"),
]

print(f"{'Finding':65s}  Layer  →  Fix")
print("-" * 95)
for title, layer in FINDINGS:
    fix = {
        "CSPM": "Change config (portal / CLI / policy).",
        "CWPP": "Investigate the alert; contain the host or principal.",
    }[layer]
    print(f"{title:65s}  {layer:5s}  →  {fix}")


## 2. Secure Score — how it's actually computed

Each recommendation has:

* A **max score** (0–10) based on severity / blast radius.
* A set of **in-scope resources** and a **healthy/unhealthy** count per resource.

Each **security control** (a grouping like "Enable MFA") scores:

```
control_score  =  max_score  ×  (healthy_resources / total_resources)
subscription_score  =  sum(control_scores)  /  sum(max_scores)  ×  100
```

So a control with 10 resources, 7 healthy and a max score of 8 is worth `8 × 0.7 = 5.6`. Controls with 0 in-scope resources are ignored — that's why enabling a new workload can *drop* your Secure Score temporarily.

In [ ]:
CONTROLS = [
    {"name": "Enable MFA",                       "max": 10, "healthy": 7,  "total": 10, "severity": "High"},
    {"name": "Restrict storage network access",  "max": 8,  "healthy": 2,  "total": 6,  "severity": "High"},
    {"name": "Enable JIT VM access",             "max": 6,  "healthy": 0,  "total": 4,  "severity": "High"},
    {"name": "Remediate SQL vulnerabilities",    "max": 4,  "healthy": 3,  "total": 5,  "severity": "High"},
    {"name": "Attach NSG to every subnet",       "max": 6,  "healthy": 8,  "total": 8,  "severity": "Medium"},
    {"name": "Enable diagnostic logs on KV",     "max": 3,  "healthy": 1,  "total": 4,  "severity": "Low"},
    {"name": "Enable DDoS Protection",           "max": 8,  "healthy": 0,  "total": 2,  "severity": "Medium"},
    {"name": "Resolve container image CVEs",     "max": 5,  "healthy": 20, "total": 25, "severity": "High"},
]


def score_controls(controls):
    earned = 0
    possible = 0
    for c in controls:
        if c["total"] == 0:
            continue  # no in-scope resources → control ignored
        earned  += c["max"] * (c["healthy"] / c["total"])
        possible += c["max"]
    return earned, possible


earned, possible = score_controls(CONTROLS)
pct = earned / possible * 100
print(f"=== Secure Score: {earned:.1f} / {possible} ({pct:.0f}%) ===\n")

print("Controls ranked by points left on the table (biggest quick wins first):")
ranked = sorted(
    CONTROLS,
    key=lambda c: c["max"] * (1 - c["healthy"] / c["total"]) if c["total"] else 0,
    reverse=True,
)
for c in ranked:
    if c["total"] == 0:
        continue
    missing = c["max"] * (1 - c["healthy"] / c["total"])
    icon = {"High": "🔴", "Medium": "🟡", "Low": "🟢"}[c["severity"]]
    print(f"  {icon} {c['name']:35s}  +{missing:4.1f} pts  ({c['healthy']}/{c['total']} healthy)")


### Exam traps

* **Exemptions remove points from *both* sides of the fraction.** If you exempt a resource
  (or disable a recommendation via its policy), it leaves the *healthy* count **and** the
  *total* count. It does **not** silently gift you a higher score, and it does not park
  the recommendation in "possible but unearned". Exemptions are audited — use them for
  genuine, documented exceptions, not to make the dashboard look nicer.
* **Adding a new subscription or workload can *lower* your Secure Score**, because
  previously out-of-scope controls suddenly have unhealthy resources in scope.
* **Secure Score is a ratio, not a count.** Fixing 1 of 2 unhealthy resources in an
  8-point control earns 4 points; fixing 1 of 200 earns almost nothing. Rank remediation
  by *points-per-effort*, which is what the ranking below computes.
* The paid **Defender CSPM** plan (not "DCSPM") unlocks *attack path analysis*, the
  *cloud security graph*, *agentless machine and container scanning*, and *data-aware
  posture management* — all of these are **CSPM**, not CWPP. The free foundational CSPM
  gives you recommendations and Secure Score only.

## 3. Defender plan advisor

There's one plan per workload type. Recommend the right one given what the customer runs.

In [ ]:
PLANS = {
    "VirtualMachines":   "Defender for Servers (P1 cheap EDR / P2 adds JIT, FIM, agentless, VA)",
    "SqlServers":        "Defender for SQL",
    "StorageAccounts":   "Defender for Storage V2 (malware scanning, sensitive-data discovery)",
    "KeyVaults":         "Defender for Key Vault",
    "AppServices":       "Defender for App Service",
    "Containers":        "Defender for Containers",
    "CosmosDbs":         "Defender for Cosmos DB",
    "Dns":               "Defender for DNS (retired as a standalone plan — DNS protection now ships inside Defender for Servers Plan 2)",
    "Arm":               "Defender for Resource Manager",
    "Apis":              "Defender for APIs",
    "OpenSourceRelationalDatabases": "Defender for open-source relational databases (PostgreSQL / MySQL / MariaDB)",
    "CloudPosture":      "Defender CSPM (attack paths, security graph, agentless scanning) — posture, not workload",
}

workloads = [
    {"type": "VirtualMachines",   "public":  True,  "internet_facing": True,  "mgmt_port_open": True},
    {"type": "SqlServers",        "critical": True},
    {"type": "StorageAccounts",   "stores_sensitive_data": True},
    {"type": "Containers",        "aks": True},
    {"type": "KeyVaults",         "count": 5},
]


def recommend_plan(w):
    plan = PLANS.get(w["type"], "No Defender plan for this workload type.")
    tier = ""
    if w["type"] == "VirtualMachines":
        # Plan 1 = Defender for Endpoint integration + agentless malware scanning.
        # Plan 2 adds JIT, File Integrity Monitoring, agentless vulnerability
        # assessment & secret scanning, adaptive application controls,
        # network hardening, 500 MB/day free data ingestion, and DNS protection.
        if w.get("mgmt_port_open") or w.get("internet_facing"):
            tier = " → Plan 2 (adds JIT, FIM, agentless VA + secret scanning, DNS protection)"
        else:
            tier = " → Plan 1 (Defender for Endpoint integration / EDR only)"
    return plan + tier


for w in workloads:
    print(f"• {w['type']:20s} {w}")
    print(f"    → {recommend_plan(w)}\n")


## 4. Cost estimator

Rough public-list prices (East US, 2024; always check your region and contract). Good enough to compare *shapes*, not to sign POs from.

In [ ]:
# Approx list prices in USD — DO NOT use for billing. Illustrative only.
PRICES = {
    "Servers P1":    5,      # per server / month
    "Servers P2":    15,     # per server / month
    "SQL":           15,     # per SQL server or Azure SQL DB vCore bundle
    "Storage V2":    10,     # per 1M transactions-ish, simplified
    "KeyVault":      2,      # per 10K transactions
    "Containers":    7,      # per vCore / month
    "AppService":    15,     # per App Service instance / month
    "DNS":           0.5,    # per 1M queries
    "ResourceMgr":   4,      # per subscription / month
}

inventory = [
    ("Servers P2",  40),
    ("SQL",         5),
    ("Storage V2",  60),
    ("KeyVault",    10),
    ("Containers",  80),
    ("AppService",  12),
    ("ResourceMgr", 1),
]

total = 0
print(f"{'Plan':15s} {'Units':>6s} {'$/unit':>9s} {'Monthly':>10s}")
print("-" * 45)
for plan, units in inventory:
    cost = units * PRICES[plan]
    total += cost
    print(f"{plan:15s} {units:>6d} {PRICES[plan]:>8.2f}  {cost:>9.2f}")
print("-" * 45)
print(f"{'TOTAL':15s} {'':>6s} {'':>8s}  ${total:>8.2f}/month  →  ~${total*12:,.0f}/year")


## 5. Regulatory compliance — mapping controls to standards

Defender for Cloud maps each policy in an initiative to one or more standards (MCSB, CIS, NIST, ISO, PCI, HIPAA HITRUST, SOC2). A **failed control in a standard** = all of the controls's policies have at least one non-compliant resource.

Below we reuse our mini-MCSB idea and compute a mini compliance report.

In [ ]:
# Each control is satisfied only if all of its policies are compliant.
CONTROLS = [
    {"id": "NS-1",  "title": "Establish network segmentation boundaries",
     "policies": ["subnet-has-nsg", "vnet-ddos-enabled"]},
    {"id": "DP-3",  "title": "Encrypt sensitive data in transit",
     "policies": ["storage-https-only", "storage-tls12"]},
    {"id": "IM-1",  "title": "Use centralized identity and authentication system",
     "policies": ["mfa-on-owners", "no-guest-admins"]},
]

# Compliance of each underlying policy (True = all resources compliant)
policy_state = {
    "subnet-has-nsg":    False,
    "vnet-ddos-enabled": False,
    "storage-https-only": True,
    "storage-tls12":     False,
    "mfa-on-owners":     True,
    "no-guest-admins":   True,
}

STANDARDS = {
    "MCSB":    ["NS-1", "DP-3", "IM-1"],
    "CIS-2.0": ["DP-3", "IM-1"],
    "PCI-4":   ["NS-1", "DP-3"],
}


def control_passes(control):
    return all(policy_state[p] for p in control["policies"])


def standard_report(name):
    ids = STANDARDS[name]
    controls = [c for c in CONTROLS if c["id"] in ids]
    passed = [c for c in controls if control_passes(c)]
    pct = len(passed) / len(controls) * 100 if controls else 100
    return passed, controls, pct


for std in STANDARDS:
    passed, controls, pct = standard_report(std)
    print(f"=== {std}: {len(passed)}/{len(controls)} controls pass ({pct:.0f}%) ===")
    for c in controls:
        icon = "✅" if control_passes(c) else "❌"
        print(f"  {icon} {c['id']}  {c['title']}")
    print()


## 6. Multi-cloud — connecting AWS and GCP

Defender for Cloud can read configuration & workload telemetry from AWS accounts and GCP projects.

```bash
# AWS — CSPM only (free) or add CWPP (Servers, SQL, Containers)
az security security-connector create -g rg-prod -n aws-connector \
  --environment-name AWS \
  --hierarchy-identifier <aws-account-id> \
  --offerings '[{"offeringType": "CspmMonitorAws"}]'

# GCP project
az security security-connector create -g rg-prod -n gcp-connector \
  --environment-name GCP \
  --hierarchy-identifier <gcp-project-number> \
  --offerings '[{"offeringType": "CspmMonitorGcp"}]'
```

**Exam trap**: Connecting AWS uses an IAM role (not an access key) via CloudFormation that Defender generates. GCP uses a service account with Security Command Center reader.

## 7. EASM — External Attack Surface Management

EASM starts from a **seed** (a domain, ASN, email) and crawls outward to map what an attacker sees: subdomains, IPs, certificates, open ports, exposed services, leaked credentials in code repos.

```bash
az easm workspace create -g rg-prod -n easm-contoso --location eastus
az easm discovery-group create -g rg-prod --workspace-name easm-contoso \
  -n contoso-discovery --seeds '[{"kind":"domain","name":"contoso.com"}]'
```

EASM findings include: unknown subdomains (shadow IT), expired TLS certificates, CVEs on exposed services, misconfigured DNS (dangling CNAMEs), and assets still pointing at deleted cloud resources.

## 8. Bad → best rollout

```bash
# ❌ BAD — free tier only, nothing enabled, no workflow automation.
#   CSPM findings pile up, no runtime protection, no alerts go anywhere.

# 🟡 BETTER — enable the plans that cover your biggest workloads.
az security pricing create -n VirtualMachines --tier Standard --subplan P2
az security pricing create -n SqlServers     --tier Standard
az security pricing create -n StorageAccounts --tier Standard --subplan DefenderForStorageV2
az security pricing create -n KeyVaults       --tier Standard

# 🟢 BEST — all workloads + DCSPM + MDE auto-provision + workflow automation + MCSB assigned.
az security pricing create -n CloudPosture  --tier Standard      # DCSPM attack-path + agentless
az security auto-provisioning-setting update --name default --auto-provision On
az policy assignment create --name mcsb \
  --policy-set-definition /providers/Microsoft.Authorization/policySetDefinitions/1f3afdf9-d0c9-4c3d-847f-89da613e70a8 \
  --scope /subscriptions/<sub-id>
az security automation create -g rg-security -n auto-high-sev ...   # triggers Logic App on High alerts
```

### One-screen summary

| Implementation       | Key details                                                                            |
|----------------------|----------------------------------------------------------------------------------------|
| CSPM vs CWPP         | CSPM = misconfig + score. CWPP = runtime alerts.                                        |
| Secure Score math    | `Σ max × (healthy/total)` ÷ `Σ max`. Controls with 0 resources are ignored.            |
| Defender plans       | One per workload type. Servers has P1 (EDR) & P2 (JIT/FIM/VA/agentless).               |
| DCSPM                | Paid CSPM. Adds attack path analysis + agentless scanning + sensitive-data discovery.  |
| Compliance           | MCSB default. Control passes only if **all** underlying policies pass.                 |
| Multi-cloud          | AWS (IAM role) & GCP (service account) via security connectors.                         |
| EASM                 | Outside-in recon. Seed with domains/ASNs.                                              |

**Next**: [Notebook 3 — Sentinel Implementation](03_sentinel_implementation.ipynb)

---
## ✅ Self-check

1. "Storage account allows public blob access" and "crypto-mining process on vm-01" — which is
   CSPM and which is CWPP, and which one costs you money to see?
2. You remediate 3 of 10 unhealthy VMs in a control worth 8 points. How many points do you earn?
3. Your Secure Score dropped after onboarding a new subscription and you changed nothing.
   Explain.
4. A customer wants **JIT VM access** and **file integrity monitoring**. Which plan and which
   tier?
5. The security team asks for "attack path analysis" and "agentless VM scanning". Which plan is
   that, and is it CSPM or CWPP?
6. Someone exempts 50 non-compliant resources to hit their Secure Score target. Does the score
   go up?

In [ ]:
answers = """
1. Public blob access = CSPM (a misconfiguration found by evaluating resource config;
   surfaces as a RECOMMENDATION and affects Secure Score). Crypto-mining on vm-01 =
   CWPP (runtime behaviour; surfaces as a security ALERT). Foundational CSPM is FREE;
   the CWPP alert requires the paid Defender for Servers plan.

2. 8 * (3/10) = 2.4 points. Secure Score is proportional: a control pays out in
   proportion to healthy/total resources, it is not all-or-nothing.

3. New subscription -> new in-scope resources -> controls that previously had zero
   in-scope resources (and were therefore excluded from the calculation) now have
   unhealthy resources counted in the denominator. Your absolute earned points may be
   unchanged while the possible points went up, so the percentage fell.

4. Microsoft Defender for Servers, PLAN 2. Plan 1 gives you the Defender for Endpoint
   integration (EDR) and agentless malware scanning; JIT, FIM, agentless vulnerability
   assessment and secret scanning, adaptive application controls and DNS protection
   are all Plan 2.

5. DEFENDER CSPM (the paid posture plan). It is CSPM -- posture and attack-path
   reasoning over configuration and the cloud security graph -- not workload runtime
   protection. Note agentless scanning also appears in Defender for Servers P2; the
   cloud security graph and attack paths are what Defender CSPM adds.

6. Only as a ratio artefact, and not in the way they hope. Exempted resources leave
   BOTH the healthy count and the total count, so a control with 50 exempted unhealthy
   resources and nothing else drops out of the calculation entirely. It is not free
   points, it is audited, and it hides real risk. Fix the resources instead.
"""
print(answers)